# 219-ROI Parcellation Pipeline
## MDD rtfMRI-NF Study — Schaefer-200 + Melbourne Subcortical

Converts preprocessed AFNI BOLD volumes (`.BRIK/.HEAD`) into 219-ROI time-series CSVs.

### Atlas
| Component | Source | ROIs |
|-----------|--------|------|
| Cortical | Schaefer 2018 (7-network, 2 mm MNI) | 200 |
| Subcortical | Melbourne / Tian et al. 2020 (Scale I, 3T) | 16 |
| Brain stem + cerebellum | Melbourne extended | 3 |
| **Total** | | **219** |

### Input structure
```
data/source/
  processed rest scans/
    E3746/           ← 5-char ID folders (rest1)
      *.BRIK  *.HEAD  *censor.1D  *enorm.1D  motion_demean.1D ...
  processed rest2 scans/
    AC_E5694_rest2.results/   ← long name (rest2), same 5-char ID inside
      *.BRIK  *.HEAD  *censor.1D  ...
```

### Output
```
data/parcellated/219roi/
  rest1/  <full_id>_rest1_219roi.csv   — 260 rows × 219 ROI columns
  rest2/  <full_id>_rest2_219roi.csv
  atlas_219roi_combined.nii.gz
  atlas_219roi_labels.csv
  parcellation_qc.csv
  parcellation_log.txt
```

### Signal notes
- Input is the AFNI **errts** file: residuals after nuisance regression, already detrended, motion-corrected, and band-pass filtered (0.01–0.1 Hz) by `afni_proc.py`.
- Censored TRs (censor.1D == 0) are set to **NaN** to preserve the temporal index. Downstream R code handles NaN via pairwise-complete observations.
- ROI signal = **mean** across all parcel voxels, no additional standardisation (handled in `mou_mvar_analysis.R`).

---

## §1 — Dependencies

Run the cell below once to install required packages. Skip if already installed.

In [1]:
# Install if needed — comment out after first run
# !pip install nibabel nilearn numpy pandas scipy tqdm

import sys
required = ['nibabel', 'nilearn', 'numpy', 'pandas', 'scipy', 'tqdm']
missing  = []
for pkg in required:
    try:
        __import__(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f'MISSING packages: {missing}')
    print('Run: pip install ' + ' '.join(missing))
else:
    print('All dependencies present.')

All dependencies present.


/opt/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


## §2 — Imports

In [2]:
import os
import re
import glob
import logging
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from scipy.ndimage import zoom

import nibabel as nib
from nilearn import image, datasets
from nilearn.maskers import NiftiLabelsMasker

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

print(f'nibabel  {nib.__version__}')
import nilearn; print(f'nilearn  {nilearn.__version__}')
print(f'numpy    {np.__version__}')
print(f'pandas   {pd.__version__}')

nibabel  5.4.2
nilearn  0.13.1
numpy    1.26.4
pandas   3.0.1


## §3 — Configuration

**Edit this cell before running.** All paths and parameters are defined here.

> `ATLAS_MELBOURNE_PATH` must point to `Tian_Subcortex_S1_3T_2009cAsym_2mm.nii.gz`  
> Download from: https://github.com/yetianmed/subcortex  
> Path: `Group-Parcellation/3T/Subcortex-Only/Tian_Subcortex_S1_3T_2009cAsym_2mm.nii.gz`

In [3]:
# ── Data paths ────────────────────────────────────────────────────────────
DATA_SOURCE_ROOT  = Path('data/source')
REST1_SOURCE_DIR  = DATA_SOURCE_ROOT / 'processed rest scans'
REST2_SOURCE_DIR  = DATA_SOURCE_ROOT / 'processed rest2 scans'
OUTPUT_DIR        = Path('data/parcellated/219roi')
LOG_FILE          = OUTPUT_DIR / 'parcellation_log.txt'

# ── Atlas paths ────────────────────────────────────────────────────────────
# Schaefer is fetched automatically via nilearn (cached to ~/.nilearn/).
# Melbourne: download and place at path below.
ATLAS_MELBOURNE_PATH = Path('atlases/Tian_Subcortex_S1_3T_2009cAsym.nii.gz')

# ── Acquisition parameters ─────────────────────────────────────────────────
TR               = 2.0    # repetition time (seconds)
N_TRS_EXPECTED   = 260    # expected volumes per scan
N_ROIS_EXPECTED  = 216    # Schaefer-200 + Melbourne-16 (Scale I)

# ── Schaefer atlas settings ────────────────────────────────────────────────
SCHAEFER_N_ROIS   = 200   # options: 100 200 300 400 500 600 800 1000
SCHAEFER_NETWORKS = 7     # 7 or 17
SCHAEFER_RES_MM   = 2     # 1 or 2 mm

# ── Processing options ─────────────────────────────────────────────────────
# NaN censored TRs (True) or drop them (False).
# True preserves temporal index — recommended for mou_mvar_analysis.R.
NAN_CENSORED_TRS   = True

# Standardise ROI time series to zero-mean unit-variance after extraction.
# Keep False — mou_mvar_analysis.R handles standardisation.
STANDARDISE_OUTPUT = False

# Skip session if more than this fraction of TRs are censored.
MIN_GOOD_TR_FRAC   = 0.80  # at most 20% censored

# ── Session filter (used in §8 processing loop) ────────────────────────────
# 'both' | 'rest1' | 'rest2'
SESSION_FILTER = 'both'

# Subject filter — set to None for all, or a 5-char ID string e.g. 'E5694'
SUBJECT_FILTER = None

print('Configuration loaded.')
print(f'  REST1 source : {REST1_SOURCE_DIR}')
print(f'  REST2 source : {REST2_SOURCE_DIR}')
print(f'  Output dir   : {OUTPUT_DIR}')
print(f'  Melbourne    : {ATLAS_MELBOURNE_PATH}  exists={ATLAS_MELBOURNE_PATH.exists()}')

Configuration loaded.
  REST1 source : data/source/processed rest scans
  REST2 source : data/source/processed rest2 scans
  Output dir   : data/parcellated/219roi
  Melbourne    : atlases/Tian_Subcortex_S1_3T_2009cAsym.nii.gz  exists=True


## §4 — Logging setup

Writes to both the notebook output and `parcellation_log.txt`.

In [4]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'rest1').mkdir(exist_ok=True)
(OUTPUT_DIR / 'rest2').mkdir(exist_ok=True)

# Clear any existing handlers
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format='%(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE, mode='w'),
        logging.StreamHandler()
    ]
)
log = logging.getLogger()

def banner(title):
    log.info('')
    log.info('=' * 70)
    log.info(title)
    log.info('=' * 70)

def section(title):
    log.info(f'\n--- {title} ---')

banner(f'219-ROI PARCELLATION PIPELINE  —  {datetime.now():%Y-%m-%d %H:%M}')
log.info(f'  Schaefer : {SCHAEFER_N_ROIS} ROIs, {SCHAEFER_NETWORKS} networks, {SCHAEFER_RES_MM}mm')
log.info(f'  TR={TR}s  N_TRS={N_TRS_EXPECTED}  NaN_censored={NAN_CENSORED_TRS}')
log.info(f'  Min good TR fraction: {MIN_GOOD_TR_FRAC:.0%}')


219-ROI PARCELLATION PIPELINE  —  2026-03-19 00:16
  Schaefer : 200 ROIs, 7 networks, 2mm
  TR=2.0s  N_TRS=260  NaN_censored=True
  Min good TR fraction: 80%


## §5 — Subject discovery

Scans both source directories and builds a subject table.

- **Rest1** folders are named with the 5-char exam ID directly: `E3746`
- **Rest2** folders encode the ID in a longer name: `AC_E5694_rest2.results`

The regex `E\d{4}` extracts the 5-char key for matching across sessions.

In [5]:
def extract_5char_id(folder_name: str) -> str | None:
    """Extract E#### pattern from any folder name."""
    match = re.search(r'E\d{4}', folder_name)
    return match.group(0) if match else None


def discover_subjects(subject_filter=None) -> dict:
    """
    Returns:
        { '5char_id': {'rest1': Path, 'rest2': Path, 'full_id': str} }
    full_id is the long prefix used in rest2 folder names (e.g. 'AC_E5694').
    """
    subjects = {}

    # rest1 ── folders are just the 5-char ID
    for folder in sorted(REST1_SOURCE_DIR.iterdir()):
        if not folder.is_dir(): continue
        sid = extract_5char_id(folder.name)
        if sid:
            subjects.setdefault(sid, {})['rest1'] = folder
            subjects[sid].setdefault('full_id', sid)

    # rest2 ── folders have long names; extract ID and prefix
    for folder in sorted(REST2_SOURCE_DIR.iterdir()):
        if not folder.is_dir(): continue
        sid = extract_5char_id(folder.name)
        if sid:
            subjects.setdefault(sid, {})['rest2'] = folder
            prefix = re.match(r'^([A-Z]{2}_E\d{4})', folder.name)
            if prefix:
                subjects[sid]['full_id'] = prefix.group(1)

    # Apply filter
    if subject_filter:
        subjects = {k: v for k, v in subjects.items() if subject_filter in k}

    return subjects


subjects = discover_subjects(SUBJECT_FILTER)

section('Subject discovery')
log.info(f'  Found {len(subjects)} subjects')
for sid, info in sorted(subjects.items()):
    sessions = [s for s in ('rest1', 'rest2') if s in info]
    log.info(f'  {sid} ({info.get("full_id", sid):>16}): {" + ".join(sessions)}')

# Summary table
df_subjects = pd.DataFrame([
    {'5char_id': sid,
     'full_id':  info.get('full_id', sid),
     'rest1':    info.get('rest1', pd.NA),
     'rest2':    info.get('rest2', pd.NA)}
    for sid, info in sorted(subjects.items())
])
print(f'\n{len(df_subjects)} subjects, '
      f'{df_subjects.rest1.notna().sum()} rest1, '
      f'{df_subjects.rest2.notna().sum()} rest2')
df_subjects[['5char_id','full_id']]


--- Subject discovery ---
  Found 23 subjects
  E3746 (        DP_E3746): rest1 + rest2
  E3799 (        HL_E3799): rest1 + rest2
  E3834 (        TP_E3834): rest1 + rest2
  E3973 (        LD_E3973): rest1 + rest2
  E4051 (        SA_E4051): rest1 + rest2
  E4209 (        LS_E4209): rest1 + rest2
  E4253 (        SA_E4253): rest1 + rest2
  E4324 (        JV_E4324): rest1 + rest2
  E4350 (        SF_E4350): rest1 + rest2
  E4360 (        KS_E4360): rest1 + rest2
  E4484 (        CB_E4484): rest1 + rest2
  E4673 (        CL_E4673): rest1 + rest2
  E4689 (        CC_E4689): rest1 + rest2
  E4696 (        DH_E4696): rest1 + rest2
  E4697 (        CM_E4697): rest1 + rest2
  E4745 (        LA_E4745): rest1 + rest2
  E5215 (        RS_E5215): rest1 + rest2
  E5376 (        KE_E5376): rest1 + rest2
  E5580 (        IR_E5580): rest1 + rest2
  E5586 (        PS_E5586): rest1 + rest2
  E5653 (        MM_E5653): rest1 + rest2
  E5693 (        KH_E5693): rest1 + rest2
  E5694 (        AC_E5694): r


23 subjects, 23 rest1, 23 rest2


,5char_id,full_id
0,E3746,DP_E3746
1,E3799,HL_E3799
2,E3834,TP_E3834
3,E3973,LD_E3973
4,E4051,SA_E4051
5,E4209,LS_E4209
6,E4253,SA_E4253
7,E4324,JV_E4324
8,E4350,SF_E4350
9,E4360,KS_E4360


## §6 — File utilities

Functions to find the BRIK file and read the AFNI motion/censor 1D files.

### censor.1D format
A single-column text file with one integer per TR:
- `1` = keep this TR
- `0` = censor this TR (excessive motion or outlier)

This file is produced by `afni_proc.py` using the enorm threshold set during preprocessing.

### enorm.1D
Euclidean norm of the motion parameter derivatives — a framewise displacement surrogate. Read here for QC reporting only; the actual censoring decisions come from `censor.1D`.

In [6]:
def find_brik_file(folder: Path) -> Path | None:
    """Return the preprocessed BRIK. Prefers 'errts' (residuals) if multiple."""
    briks = sorted(folder.glob('*.BRIK')) + sorted(folder.glob('*.BRIK.gz'))
    if not briks:
        log.warning(f'  No .BRIK file in {folder}')
        return None
    if len(briks) > 1:
        errts = [b for b in briks if 'errts' in b.name.lower()]
        if errts:
            log.info(f'  Multiple BRIKs — using errts: {errts[0].name}')
            return errts[0]
        log.warning(f'  Multiple BRIKs, no errts match — using: {briks[0].name}')
    return briks[0]


def read_censor(folder: Path) -> np.ndarray | None:
    """
    Load *censor.1D  →  bool array (True = keep).
    Returns None if file is absent (all TRs treated as good).
    """
    files = sorted(folder.glob('*censor.1D'))
    if not files:
        log.warning(f'  No censor.1D in {folder.name} — assuming all TRs good')
        return None
    vec = np.loadtxt(files[0]).astype(bool)
    return vec


def read_enorm(folder: Path) -> np.ndarray | None:
    """Load *enorm.1D for QC reporting (not used for censoring)."""
    files = sorted(folder.glob('*enorm.1D'))
    return np.loadtxt(files[0]) if files else None


def read_motion(folder: Path) -> dict:
    """
    Read motion_demean.1D and motion_deriv.1D for QC.
    Returns dict with mean absolute motion per parameter.
    """
    result = {}
    for fname, key in [('motion_demean.1D', 'demean'), ('motion_deriv.1D', 'deriv')]:
        fpath = folder / fname
        if fpath.exists():
            mat = np.loadtxt(fpath)
            result[f'{key}_mean_abs'] = float(np.mean(np.abs(mat)))
            result[f'{key}_max_abs']  = float(np.max(np.abs(mat)))
    return result


print('File utility functions defined.')

File utility functions defined.


## §7 — Atlas construction

### §7a — Load Schaefer cortical atlas
Downloaded automatically via nilearn on first call, then cached at `~/.nilearn/`.

### §7b — Load Melbourne subcortical atlas
Must be downloaded manually (see §3 note). Scale I contains 16 bilateral subcortical structures.

### §7c — Combine into 219-ROI atlas
Melbourne parcels are resampled to the Schaefer 2 mm MNI grid using nearest-neighbour interpolation (preserves integer labels), then offset to labels 201–219 so they do not overlap with Schaefer labels 1–200. Cortical parcels take priority in any voxel overlap.

In [7]:
# §7a — Schaefer
section('Loading Schaefer 2018 atlas')
schaefer = datasets.fetch_atlas_schaefer_2018(
    n_rois=SCHAEFER_N_ROIS,
    yeo_networks=SCHAEFER_NETWORKS,
    resolution_mm=SCHAEFER_RES_MM
)
schaefer_img    = nib.load(schaefer.maps)
schaefer_labels = [l.decode() if isinstance(l, bytes) else l
                   for l in schaefer.labels]

# nilearn includes 'Background' as the first label (index 0) — strip it.
# The atlas image uses 0 for background voxels; NiftiLabelsMasker already
# excludes these from extraction, but the label must be removed here so
# that atlas_labels has exactly 200 entries and indices align 1-to-1.
schaefer_labels = [l for l in schaefer_labels if l.lower() != 'background']

log.info(f'  Schaefer: {len(schaefer_labels)} ROIs (Background label stripped)')
log.info(f'  Image shape: {schaefer_img.shape}  '
         f'voxel: {schaefer_img.header.get_zooms()[:3]}')
print(f'Example labels: {schaefer_labels[:3]} ... {schaefer_labels[-3:]}')


--- Loading Schaefer 2018 atlas ---


[fetch_atlas_schaefer_2018] Dataset found in /Users/slm/nilearn_data/schaefer_2018

  Schaefer: 200 ROIs (Background label stripped)
  Image shape: (91, 109, 91)  voxel: (2.0, 2.0, 2.0)


Example labels: ['7Networks_LH_Vis_1', '7Networks_LH_Vis_2', '7Networks_LH_Vis_3'] ... ['7Networks_RH_Default_pCunPCC_1', '7Networks_RH_Default_pCunPCC_2', '7Networks_RH_Default_pCunPCC_3']


In [8]:
# §7b — Melbourne subcortical (with auto-download fallback)
section('Loading Melbourne Subcortical atlas')

def _try_download_melbourne(dest_path):
    """Try to download Melbourne Tian Scale-I 3T atlas from multiple sources."""
    import urllib.request, ssl, tempfile, shutil
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    urls = [
        # OSF direct download (Tian et al. 2020)
        "https://files.osf.io/v1/resources/jkzwp/providers/osfstorage/5f9a2a0e99f9a700a24d8e29",
        # GitHub raw (yetianmed/subcortex)
        "https://raw.githubusercontent.com/yetianmed/subcortex/master/"
        "Group-Parcellation/3T/Subcortex-Only/Tian_Subcortex_S1_3T_2009cAsym_2mm.nii.gz",
    ]
    for url in urls:
        try:
            log.info(f'  Downloading Melbourne atlas from {url[:60]}...')
            with urllib.request.urlopen(url, context=ctx, timeout=60) as resp:
                with open(dest_path, 'wb') as f:
                    shutil.copyfileobj(resp, f)
            log.info(f'  Download complete -> {dest_path}')
            return True
        except Exception as e:
            log.warning(f'  Download failed: {e}')
    return False

# Try the configured path first, then try nilearn fetcher, then download
MELBOURNE_FOUND = False
melbourne_img   = None

if ATLAS_MELBOURNE_PATH.exists():
    melbourne_img   = nib.load(ATLAS_MELBOURNE_PATH)
    MELBOURNE_FOUND = True
    log.info(f'  Melbourne atlas found at configured path.')
else:
    # Try nilearn built-in Tian fetcher (nilearn >= 0.10.3)
    try:
        from nilearn.datasets import fetch_atlas_tian_et_al_2021
        tian = fetch_atlas_tian_et_al_2021(atlas="S1",
                                            space="MNI152NLin2009cAsym",
                                            resolution=2)
        melbourne_img   = nib.load(tian.maps)
        MELBOURNE_FOUND = True
        log.info('  Melbourne atlas loaded via nilearn Tian fetcher.')
    except Exception as e:
        log.warning(f'  nilearn Tian fetcher unavailable: {e}')

    if not MELBOURNE_FOUND:
        # Try downloading
        ATLAS_MELBOURNE_PATH.parent.mkdir(parents=True, exist_ok=True)
        if _try_download_melbourne(ATLAS_MELBOURNE_PATH):
            try:
                melbourne_img   = nib.load(ATLAS_MELBOURNE_PATH)
                MELBOURNE_FOUND = True
            except Exception as e:
                log.warning(f'  Downloaded file unreadable: {e}')

if not MELBOURNE_FOUND:
    log.warning('Melbourne atlas unavailable — will produce 200-ROI (Schaefer only) output.')
    log.warning('To enable 216-ROI output, download Tian_Subcortex_S1_3T_2009cAsym_2mm.nii.gz')
    log.warning('from https://github.com/yetianmed/subcortex and place at:')
    log.warning(f'  {ATLAS_MELBOURNE_PATH.resolve()}')
    melbourne_img   = None
    melbourne_labels = []
else:
    # Scale I label ordering (Tian et al. 2020, Table S2)
    melbourne_labels = [
        'Hipp-lh', 'Hipp-rh',
        'Amyg-lh', 'Amyg-rh',
        'Thal-lh', 'Thal-rh',
        'Caud-lh', 'Caud-rh',
        'Put-lh',  'Put-rh',
        'Pall-lh', 'Pall-rh',
        'NAcc-lh', 'NAcc-rh',
        'STN-lh',  'STN-rh'
    ]
    n_mel = int(melbourne_img.get_fdata().max())
    if n_mel != len(melbourne_labels):
        log.warning(f'  Melbourne: {n_mel} parcels found, '
                    f'{len(melbourne_labels)} labels expected — using generic names')
        melbourne_labels = [f'Subcortical_{i+1}' for i in range(n_mel)]
    log.info(f'  Melbourne: {n_mel} subcortical ROIs')
    log.info(f'  Image shape: {melbourne_img.shape}  '
             f'voxel: {melbourne_img.header.get_zooms()[:3]}')
    print(f'Melbourne labels: {melbourne_labels}')


--- Loading Melbourne Subcortical atlas ---
  Melbourne atlas found at configured path.
  Melbourne: 16 subcortical ROIs
  Image shape: (97, 115, 97)  voxel: (2.0, 2.0, 2.0)


Melbourne labels: ['Hipp-lh', 'Hipp-rh', 'Amyg-lh', 'Amyg-rh', 'Thal-lh', 'Thal-rh', 'Caud-lh', 'Caud-rh', 'Put-lh', 'Put-rh', 'Pall-lh', 'Pall-rh', 'NAcc-lh', 'NAcc-rh', 'STN-lh', 'STN-rh']


In [9]:
# §7c — Combine atlases
section('Combining cortical + subcortical atlases')

if melbourne_img is not None:
    mel_resampled = image.resample_to_img(
        melbourne_img, schaefer_img, interpolation='nearest'
    )
    sch_data = np.array(schaefer_img.dataobj).astype(np.int16)
    mel_data = np.array(mel_resampled.dataobj).astype(np.int16)

    mel_offset = mel_data.copy()
    mel_offset[mel_data > 0] = mel_data[mel_data > 0] + SCHAEFER_N_ROIS

    combined = sch_data.copy()
    subcortical_mask = (mel_offset > 0) & (combined == 0)
    combined[subcortical_mask] = mel_offset[subcortical_mask]
else:
    # Schaefer-200 only fallback
    sch_data = np.array(schaefer_img.dataobj).astype(np.int16)
    combined  = sch_data.copy()

atlas_img    = nib.Nifti1Image(combined, schaefer_img.affine, schaefer_img.header)
atlas_labels = schaefer_labels + melbourne_labels

N_ROIS_ACTUAL = len(atlas_labels)

# Save combined atlas and label table
atlas_save = OUTPUT_DIR / 'atlas_219roi_combined.nii.gz'
nib.save(atlas_img, atlas_save)

labels_df = pd.DataFrame({
    'roi_index': range(1, len(atlas_labels) + 1),
    'roi_label': atlas_labels,
    'source': (['Schaefer'] * len(schaefer_labels) +
                ['Melbourne'] * len(melbourne_labels))
})
labels_df.to_csv(OUTPUT_DIR / 'atlas_219roi_labels.csv', index=False)

n_assigned = int(np.sum(combined > 0))
log.info(f'  Combined: {len(atlas_labels)} ROIs total')
log.info(f'  Voxels assigned: {n_assigned:,} ({100*n_assigned/combined.size:.1f}% of volume)')
log.info(f'  Saved: {atlas_save}')

print(f'\nAtlas ready: {len(atlas_labels)} ROIs')
print(f'First cortical:    {atlas_labels[0]}')
print(f'Last cortical:     {atlas_labels[SCHAEFER_N_ROIS-1]}')
if melbourne_labels:
    print(f'First subcortical: {atlas_labels[SCHAEFER_N_ROIS]}')
    print(f'Last subcortical:  {atlas_labels[-1]}')


--- Combining cortical + subcortical atlases ---
  Combined: 216 ROIs total
  Voxels assigned: 140,426 (15.6% of volume)
  Saved: data/parcellated/219roi/atlas_219roi_combined.nii.gz



Atlas ready: 216 ROIs
First cortical:    7Networks_LH_Vis_1
Last cortical:     7Networks_RH_Default_pCunPCC_3
First subcortical: Hipp-lh
Last subcortical:  STN-rh


## §7d — HOA Label Atlas (from atlas.R)

Constructs the 110-region HOA parcellation using a **label-image approach**:
a 3D NIfTI atlas is built by assigning each voxel to its nearest sphere centre
within the radius. Extraction then uses `NiftiLabelsMasker`, which has no
overlap restriction.

**Why not `NiftiSpheresMasker`?**  
Five sphere pairs overlap at r=5mm (closest: CALC.R–SCLC.R at 6.75mm).
`NiftiSpheresMasker` raises an error when any two seeds are < 2×radius apart.
The label-image approach resolves overlaps by winner-takes-all (nearest centre),
so all 5 affected voxels are cleanly assigned to one region without data loss.

| Property | Value |
|----------|-------|
| Regions | 110 (55 bilateral L/R pairs) |
| Method | Label NIfTI image + `NiftiLabelsMasker` |
| Overlap handling | Nearest-centre winner-takes-all |
| Source | `R/atlas.R` → `HOA_ATLAS` |
| Output | `data/parcellated/hoa_110roi/{rest1,rest2}/` |

In [10]:
# §7d — HOA sphere atlas (from R/atlas.R)
# Uses a label-image approach instead of NiftiSpheresMasker to avoid the
# "spheres overlap" error that occurs when two seeds are < 2*radius apart.
# Five pairs overlap at r=5mm (closest: CALC.R–SCLC.R at 6.75mm).
# Fix: build a 3D label image via nearest-centre assignment (winner-takes-all),
# then extract with NiftiLabelsMasker which has no overlap restriction.

from nilearn.maskers import NiftiLabelsMasker
import numpy as np

HOA_SPHERE_RADIUS = 5   # mm  (unchanged — good SNR, overlaps resolved below)

# 110 MNI coordinates and region names from R/atlas.R HOA_ATLAS
HOA_NAMES  = ['Thal.L', 'Thal.R', 'Caud.L', 'Caud.R', 'Put.L', 'Put.R', 'Pall.L', 'Pall.R', 'Hip.L', 'Hip.R', 'Amy.L', 'Amy.R', 'Accbns.L', 'Accbns.R', 'FP.L', 'FP.R', 'INS.L', 'INS.R', 'F1.L', 'F1.R', 'F2.L', 'F2.R', 'F3t.L', 'F3t.R', 'F3o.L', 'F3o.R', 'PRG.L', 'PRG.R', 'TP.L', 'TP.R', 'T1a.L', 'T1a.R', 'T1p.L', 'T1p.R', 'T2a.L', 'T2a.R', 'T2p.L', 'T2p.R', 'TO2.L', 'TO2.R', 'T3a.L', 'T3a.R', 'T3p.L', 'T3p.R', 'TO3.L', 'TO3.R', 'POG.L', 'POG.R', 'SPL.L', 'SPL.R', 'SGa.L', 'SGa.R', 'SGp.L', 'SGp.R', 'AG.L', 'AG.R', 'OLs.L', 'OLs.R', 'OLi.L', 'OLi.R', 'CALC.L', 'CALC.R', 'FMC.L', 'FMC.R', 'SMC.L', 'SMC.R', 'SC.L', 'SC.R', 'PAC.L', 'PAC.R', 'CGa.L', 'CGa.R', 'CGp.L', 'CGp.R', 'PCN.L', 'PCN.R', 'CN.L', 'CN.R', 'FOC.L', 'FOC.R', 'PHa.L', 'PHa.R', 'PHp.L', 'PHp.R', 'LG.L', 'LG.R', 'TFa.L', 'TFa.R', 'TFp.L', 'TFp.R', 'TOF.L', 'TOF.R', 'OF.L', 'OF.R', 'FO.L', 'FO.R', 'CO.L', 'CO.R', 'PO.L', 'PO.R', 'PP.L', 'PP.R', 'H.L', 'H.R', 'PT.L', 'PT.R', 'SCLC.L', 'SCLC.R', 'OP.L', 'OP.R']
HOA_LOBES  = ['SCGM', 'SCGM', 'SCGM', 'SCGM', 'SCGM', 'SCGM', 'SCGM', 'SCGM', 'SCGM', 'SCGM', 'SCGM', 'SCGM', 'SCGM', 'SCGM', 'Frontal', 'Frontal', 'Insula', 'Insula', 'Frontal', 'Frontal', 'Frontal', 'Frontal', 'Frontal', 'Frontal', 'Frontal', 'Frontal', 'Frontal', 'Frontal', 'Temporal', 'Temporal', 'Temporal', 'Temporal', 'Temporal', 'Temporal', 'Temporal', 'Temporal', 'Temporal', 'Temporal', 'Temporal', 'Temporal', 'Temporal', 'Temporal', 'Temporal', 'Temporal', 'Temporal', 'Temporal', 'Parietal', 'Parietal', 'Parietal', 'Parietal', 'Parietal', 'Parietal', 'Parietal', 'Parietal', 'Parietal', 'Parietal', 'Occipital', 'Occipital', 'Occipital', 'Occipital', 'Occipital', 'Occipital', 'Frontal', 'Frontal', 'Frontal', 'Frontal', 'Frontal', 'Frontal', 'Cingulate', 'Cingulate', 'Cingulate', 'Cingulate', 'Cingulate', 'Cingulate', 'Parietal', 'Parietal', 'Occipital', 'Occipital', 'Frontal', 'Frontal', 'Temporal', 'Temporal', 'Temporal', 'Temporal', 'Occipital', 'Occipital', 'Temporal', 'Temporal', 'Temporal', 'Temporal', 'Occipital', 'Occipital', 'Occipital', 'Occipital', 'Frontal', 'Frontal', 'Frontal', 'Frontal', 'Parietal', 'Parietal', 'Temporal', 'Temporal', 'Temporal', 'Temporal', 'Temporal', 'Temporal', 'Occipital', 'Occipital', 'Occipital', 'Occipital']
HOA_COORDS = [(-9.99,-19.16,16.28),(10.92,-18.5,16.6),(-12.84,8.92,19.71),(13.5,9.66,20.87),(-24.79,0.54,10.42),(25.48,2.03,10.35),(-19.13,-5.17,8.53),(19.75,-3.87,8.56),(-25.18,-23.25,-3.92),(26.5,-20.99,-4.07),(-22.86,-5.18,-7.49),(22.77,-3.69,-7.91),(-9.34,11.14,2.89),(9.21,11.46,3.57),(-25.01,52.77,17.75),(26.44,52.0,18.6),(-36.42,1.01,10.16),(37.5,2.65,9.83),(-14.56,17.96,66.57),(15.09,17.55,67.52),(-38.15,18.31,52.01),(39.19,18.3,53.02),(-49.76,28.6,18.59),(51.84,27.82,17.72),(-50.69,14.63,25.22),(52.49,15.48,26.37),(-34.28,-11.71,59.18),(35.08,-10.59,59.79),(-40.44,11.07,-19.78),(40.99,12.93,-19.31),(-56.0,-3.79,1.86),(57.22,-0.98,-0.41),(-62.37,-29.14,13.86),(61.35,-23.87,11.5),(-57.8,-4.41,-12.05),(57.86,-1.74,-14.52),(-60.95,-27.39,-0.91),(60.97,-22.35,-2.18),(-57.4,-52.7,10.87),(58.32,-49.3,11.53),(-47.97,-5.1,-29.12),(46.31,-2.16,-31.18),(-53.46,-28.21,-16.0),(53.8,-23.36,-18.1),(-51.81,-53.45,-6.68),(54.19,-49.71,-6.86),(-38.53,-27.78,61.5),(37.25,-26.58,62.93),(-29.28,-49.4,67.63),(29.08,-47.79,68.92),(-57.0,-32.5,46.94),(58.23,-27.26,48.18),(-54.86,-46.04,43.58),(55.23,-40.29,43.9),(-50.45,-55.74,39.3),(52.15,-51.69,42.16),(-32.05,-72.77,47.99),(33.03,-71.06,49.0),(-45.21,-75.63,8.06),(45.47,-74.11,8.49),(-10.47,-74.92,18.19),(11.93,-73.63,18.36),(-5.38,43.84,-7.87),(5.51,43.41,-8.24),(-5.79,-2.67,66.3),(6.39,-2.85,67.64),(-5.7,20.6,-5.68),(5.66,20.42,-5.9),(-6.82,36.58,30.93),(7.07,36.37,32.84),(-5.15,18.19,34.6),(5.75,19.31,34.15),(-6.3,-38.56,38.79),(6.96,-35.8,40.04),(-8.16,-60.06,47.25),(9.33,-58.48,48.1),(-8.66,-80.04,37.62),(9.38,-78.23,37.98),(-29.69,23.81,-6.49),(29.31,23.43,-6.21),(-21.68,-9.28,-20.7),(22.59,-8.04,-20.63),(-22.15,-32.31,-7.13),(22.95,-30.25,-6.98),(-12.57,-65.51,4.55),(13.93,-62.73,5.03),(-32.3,-4.53,-31.6),(30.87,-2.55,-32.28),(-36.02,-29.45,-15.04),(36.53,-23.81,-18.05),(-33.32,-53.65,-5.95),(35.02,-49.88,-6.56),(-26.33,-76.86,-3.45),(27.24,-75.48,-2.29),(-39.82,18.29,14.62),(41.14,18.82,14.75),(-48.03,-8.28,21.64),(49.47,-5.66,21.1),(-48.4,-31.53,30.3),(48.86,-27.69,31.65),(-46.77,-5.34,2.46),(48.11,-3.56,2.9),(-45.22,-20.04,17.31),(46.04,-17.36,16.89),(-52.64,-29.69,20.8),(54.84,-25.33,22.39),(-12.29,-68.86,25.57),(9.09,-74.03,24.47),(-17.23,-96.36,17.23),(18.09,-95.14,18.35)]
HOA_COORDS_ARR = np.array(HOA_COORDS)   # shape (110, 3)

HOA_OUTPUT_DIR = Path('data/parcellated/hoa_110roi')
(HOA_OUTPUT_DIR / 'rest1').mkdir(parents=True, exist_ok=True)
(HOA_OUTPUT_DIR / 'rest2').mkdir(parents=True, exist_ok=True)

# ── Report overlapping pairs so the user knows what was resolved ──────────────
_overlapping = []
for _i in range(len(HOA_COORDS_ARR)):
    for _j in range(_i+1, len(HOA_COORDS_ARR)):
        _d = np.linalg.norm(HOA_COORDS_ARR[_i] - HOA_COORDS_ARR[_j])
        if _d < 2 * HOA_SPHERE_RADIUS:
            _overlapping.append((HOA_NAMES[_i], HOA_NAMES[_j], round(_d, 2)))
if _overlapping:
    log.info(f'  {len(_overlapping)} overlapping sphere pairs at r={HOA_SPHERE_RADIUS}mm '
             f'(resolved by nearest-centre assignment):')
    for _a, _b, _d in _overlapping:
        log.info(f'    {_a} -- {_b}  dist={_d}mm')
else:
    log.info('  No overlapping sphere pairs.')


def build_sphere_atlas_img(coords_mni, radius_mm, ref_img):
    """
    Build a 3D integer label NIfTI image from sphere seed coordinates.
    Each voxel is assigned to the NEAREST seed centre whose distance is
    within radius_mm.  When two spheres overlap, the closer centre wins
    (winner-takes-all), so every voxel belongs to exactly one region.
    Background voxels (outside all spheres) keep label 0.

    Parameters
    ----------
    coords_mni : ndarray, shape (N, 3)
        Seed coordinates in MNI mm.
    radius_mm  : float
        Sphere radius in mm.
    ref_img    : Nifti1Image
        Reference image that defines the voxel grid and affine.

    Returns
    -------
    label_img : Nifti1Image
        Integer label image (1-indexed), same grid as ref_img.
    """
    affine    = ref_img.affine
    inv_aff   = np.linalg.inv(affine)
    shape     = ref_img.shape[:3]
    label_arr = np.zeros(shape, dtype=np.int16)

    # Convert MNI coords -> voxel coords
    coords_vox = nib.affines.apply_affine(inv_aff, coords_mni)  # (N, 3)

    # Voxel sizes (mm per voxel along each axis)
    vox_sizes = np.sqrt((affine[:3, :3] ** 2).sum(axis=0))      # (3,)
    radius_vox = radius_mm / vox_sizes                           # (3,)

    # Build bounding box for efficiency: only iterate over voxels that
    # could possibly be within radius of any seed
    i_grid, j_grid, k_grid = np.mgrid[
        0:shape[0], 0:shape[1], 0:shape[2]
    ]
    vox_coords = np.column_stack([i_grid.ravel(),
                                   j_grid.ravel(),
                                   k_grid.ravel()]).astype(float)   # (V, 3)

    # For each voxel compute distance (in mm) to all seeds, find nearest
    # Vectorised: (V,3) broadcast against (N,3) -> (V,N)
    # Process in chunks to avoid memory issues for large images
    CHUNK = 50000
    for start in range(0, len(vox_coords), CHUNK):
        chunk  = vox_coords[start:start+CHUNK]                  # (C, 3)
        # Distance in mm: convert voxel diff to mm via affine
        diff   = chunk[:, None, :] - coords_vox[None, :, :]     # (C, N, 3)
        mm     = (diff * vox_sizes[None, None, :])               # (C, N, 3)
        dist   = np.sqrt((mm ** 2).sum(axis=2))                  # (C, N)
        # Only consider seeds within radius
        in_rad = dist <= radius_mm                               # (C, N) bool
        # Nearest seed for each voxel (among those in radius)
        dist_masked = np.where(in_rad, dist, np.inf)
        nearest_idx  = dist_masked.argmin(axis=1)                # (C,)
        has_seed     = in_rad.any(axis=1)                        # (C,) bool

        for ci, (idx, assigned) in enumerate(zip(nearest_idx, has_seed)):
            if assigned:
                vi, vj, vk = chunk[ci].astype(int)
                label_arr[vi, vj, vk] = idx + 1  # 1-indexed

    return nib.Nifti1Image(label_arr, affine, ref_img.header)


# Build the HOA label atlas image using the Schaefer image as the reference grid
log.info('Building HOA sphere label atlas (nearest-centre, winner-takes-all)...')
_t0 = __import__('time').time()
hoa_atlas_img = build_sphere_atlas_img(HOA_COORDS_ARR, HOA_SPHERE_RADIUS, schaefer_img)
log.info(f'  Built in {__import__("time").time()-_t0:.1f}s')

# Verify: should have 110 unique non-zero labels
_unique_labels = np.unique(hoa_atlas_img.get_fdata().astype(int))
_unique_labels  = _unique_labels[_unique_labels > 0]
log.info(f'  Unique non-zero labels: {len(_unique_labels)} (expected 110)')
if len(_unique_labels) < 110:
    log.warning(f'  Missing labels: {set(range(1,111)) - set(_unique_labels.tolist())}')
    log.warning('  These regions may fall outside the reference image FOV.')

# Save HOA atlas and label table
_hoa_atlas_save = HOA_OUTPUT_DIR / 'hoa_110roi_atlas.nii.gz'
nib.save(hoa_atlas_img, _hoa_atlas_save)
log.info(f'  Saved: {_hoa_atlas_save}')

pd.DataFrame({
    'roi_index': range(1, len(HOA_NAMES)+1),
    'roi_name':  HOA_NAMES,
    'lobe':      HOA_LOBES,
    'x': HOA_COORDS_ARR[:,0],
    'y': HOA_COORDS_ARR[:,1],
    'z': HOA_COORDS_ARR[:,2]
}).to_csv(HOA_OUTPUT_DIR / 'hoa_110roi_labels.csv', index=False)

print(f'HOA atlas ready: {len(_unique_labels)}/110 regions assigned')
print(f'Output: {HOA_OUTPUT_DIR}')

  5 overlapping sphere pairs at r=5mm (resolved by nearest-centre assignment):
    Put.L -- Pall.L  dist=8.26mm
    Put.R -- Pall.R  dist=8.42mm
    T1a.L -- PP.L  dist=9.38mm
    CALC.L -- SCLC.L  dist=9.72mm
    CALC.R -- SCLC.R  dist=6.75mm
Building HOA sphere label atlas (nearest-centre, winner-takes-all)...
  Built in 2.7s
  Unique non-zero labels: 110 (expected 110)
  Saved: data/parcellated/hoa_110roi/hoa_110roi_atlas.nii.gz


HOA atlas ready: 110/110 regions assigned
Output: data/parcellated/hoa_110roi


In [12]:
def load_bold(brik_path: Path):
    """Load AFNI BRIK as nibabel image. Validates 4D shape."""
    img = nib.load(str(brik_path))
    if img.ndim != 4:
        raise ValueError(f'Expected 4D, got {img.ndim}D: {brik_path}')
    n_vols = img.shape[3]
    if n_vols != N_TRS_EXPECTED:
        log.warning(f'  Unexpected volume count: {n_vols} (expected {N_TRS_EXPECTED})')
    return img


def extract_roi_timeseries(bold_img, atlas_resampled):
    """
    Extract mean ROI time series using NiftiLabelsMasker.
    Returns (timeseries, masker) where timeseries.shape = (n_TRs, n_ROIs).
    """
    masker = NiftiLabelsMasker(
        labels_img=atlas_resampled,
        standardize=False,
        detrend=False,
        smoothing_fwhm=None,
        memory_level=0,
        verbose=0
    )
    timeseries = masker.fit_transform(bold_img)  # (n_TRs, n_ROIs)
    return timeseries, masker


def apply_censoring(timeseries: np.ndarray,
                    censor_vec: np.ndarray | None) -> tuple[np.ndarray, int]:
    """
    Replace censored TR rows with NaN (or drop, depending on NAN_CENSORED_TRS).
    Returns (timeseries_out, n_censored_applied).
    """
    if censor_vec is None:
        return timeseries, 0

    n_trs = timeseries.shape[0]
    if len(censor_vec) != n_trs:
        log.warning(f'  Censor length {len(censor_vec)} != n_TRs {n_trs} — skipping')
        return timeseries, 0

    n_censored = int(np.sum(~censor_vec))
    if NAN_CENSORED_TRS:
        out = timeseries.astype(float)
        out[~censor_vec, :] = np.nan
    else:
        out = timeseries[censor_vec, :]

    return out, n_censored


def build_column_names(masker, atlas_labels: list) -> list:
    """
    Match masker-returned label integers back to atlas label strings.
    Excludes label 0 (background) which NiftiLabelsMasker drops from
    extraction but may still include in masker.labels_.
    """
    returned = [int(x) for x in masker.labels_ if int(x) != 0]
    cols = []
    for li in returned:
        if 1 <= li <= len(atlas_labels):
            cols.append(atlas_labels[li - 1])
        else:
            cols.append(f'ROI_{li}')
    return cols


print('Extraction functions defined.')

Extraction functions defined.


### §7d pilot — verify HOA extraction on one subject before batch

In [13]:
# §7d pilot — test HOA extraction on first subject
section('HOA label-atlas extraction — pilot test')

def load_bold(brik_path: Path):
    """Load AFNI BRIK as nibabel image. Validates 4D shape."""
    img = nib.load(str(brik_path))
    if img.ndim != 4:
        raise ValueError(f'Expected 4D, got {img.ndim}D: {brik_path}')
    n_vols = img.shape[3]
    if n_vols != N_TRS_EXPECTED:
        log.warning(f'  Unexpected volume count: {n_vols} (expected {N_TRS_EXPECTED})')
    return img

_test_sid  = sorted(subjects.keys())[0]
_test_info = subjects[_test_sid]
_test_sess = 'rest1' if 'rest1' in _test_info else 'rest2'
_test_brik = find_brik_file(_test_info[_test_sess])

_bold = load_bold(_test_brik)

# Resample HOA label atlas to BOLD space, then extract with NiftiLabelsMasker
_hoa_resampled = image.resample_to_img(hoa_atlas_img, _bold, interpolation='nearest')

hoa_masker_test = NiftiLabelsMasker(
    labels_img=_hoa_resampled,
    standardize=False,
    detrend=False,
    smoothing_fwhm=None,
    memory_level=0,
    verbose=0
)
_ts_hoa = hoa_masker_test.fit_transform(_bold)

# Map masker output columns back to HOA_NAMES
_returned_labels = [int(x) for x in hoa_masker_test.labels_ if int(x) != 0]
_col_names = [HOA_NAMES[l-1] if 1 <= l <= len(HOA_NAMES) else f'HOA_{l}'
              for l in _returned_labels]

print(f'HOA extraction shape : {_ts_hoa.shape}  (TRs x regions)')
print(f'Expected             : ({_bold.shape[3]}, {len(HOA_NAMES)})')
print(f'Regions extracted    : {len(_col_names)}')
print(f'NaN count            : {np.isnan(_ts_hoa).sum()}')
print(f'First 4 columns      : {_col_names[:4]}')

_censor = read_censor(_test_info[_test_sess])
_ts_c, _nc = apply_censoring(_ts_hoa, _censor)
print(f'After censoring      : {_ts_c.shape}  censored_TRs={_nc}')
print('HOA pilot: OK')


--- HOA label-atlas extraction — pilot test ---


HOA extraction shape : (260, 110)  (TRs x regions)
Expected             : (260, 110)
Regions extracted    : 110
NaN count            : 0
First 4 columns      : ['Thal.L', 'Thal.R', 'Caud.L', 'Caud.R']
After censoring      : (260, 110)  censored_TRs=26
HOA pilot: OK


### §7d full batch — HOA extraction all subjects × sessions

In [15]:
# §7d full batch — HOA label-atlas extraction, all subjects x sessions
OVERWRITE_HOA = False

section('HOA full batch (label-atlas extraction)')
banner('HOA EXTRACTION — FULL BATCH')
sessions_to_run = (['rest1', 'rest2'] if SESSION_FILTER == 'both'
                   else [SESSION_FILTER])
hoa_qc = []

for sid, info in sorted(subjects.items()):
    full_id = info.get('full_id', sid)

    for sess in sessions_to_run:
        if sess not in info:
            continue

        out_path = HOA_OUTPUT_DIR / sess / f'{full_id}_{sess}_hoa110roi.csv'
        if out_path.exists() and not OVERWRITE_HOA:
            log.info(f'  {full_id} {sess} HOA: already done — skipping')
            hoa_qc.append({'full_id': full_id, 'session': sess,
                            'status': 'SKIPPED', 'n_rois': 0, 'notes': ''})
            continue

        log.info(f'  HOA [{sess.upper()}] {full_id}')
        qr = {'full_id': full_id, 'session': sess,
              'status': 'FAIL', 'n_rois': 0, 'notes': ''}

        try:
            brik = find_brik_file(info[sess])
            if brik is None:
                qr['notes'] = 'No BRIK'; hoa_qc.append(qr); continue

            bold     = load_bold(brik)
            censor_v = read_censor(info[sess])

            # Resample HOA atlas to this subject's BOLD space
            hoa_res = image.resample_to_img(
                hoa_atlas_img, bold, interpolation='nearest'
            )

            # Extract using NiftiLabelsMasker (no overlap restriction)
            masker = NiftiLabelsMasker(
                labels_img=hoa_res,
                standardize=False,
                detrend=False,
                smoothing_fwhm=None,
                memory_level=0,
                verbose=0
            )
            ts = masker.fit_transform(bold)   # (TRs, N_regions_found)

            # Map returned label indices -> HOA_NAMES
            ret_labels = [int(x) for x in masker.labels_ if int(x) != 0]
            col_names  = [HOA_NAMES[l-1] if 1 <= l <= len(HOA_NAMES)
                          else f'HOA_{l}' for l in ret_labels]

            ts, _ = apply_censoring(ts, censor_v)

            df_hoa = pd.DataFrame(ts, columns=col_names)
            df_hoa.index.name = 'TR'
            df_hoa.to_csv(out_path)

            qr['n_rois'] = ts.shape[1]; qr['status'] = 'OK'
            log.info(f'    Saved {ts.shape[0]}x{ts.shape[1]} -> {out_path.name}')

        except Exception as e:
            log.error(f'    ERROR: {e}')
            qr['notes'] = str(e)

        hoa_qc.append(qr)

hoa_qc_df = pd.DataFrame(hoa_qc)
hoa_qc_df.to_csv(HOA_OUTPUT_DIR / 'hoa_parcellation_qc.csv', index=False)
log.info(f'HOA batch done: {(hoa_qc_df.status=="OK").sum()} OK  '
         f'{(hoa_qc_df.status=="FAIL").sum()} FAIL  '
         f'{(hoa_qc_df.status=="SKIPPED").sum()} skipped')
print(hoa_qc_df[['full_id','session','status','n_rois']].to_string(index=False))


--- HOA full batch (label-atlas extraction) ---

HOA EXTRACTION — FULL BATCH
  HOA [REST1] DP_E3746
    Saved 260x110 -> DP_E3746_rest1_hoa110roi.csv
  HOA [REST2] DP_E3746
    Saved 260x110 -> DP_E3746_rest2_hoa110roi.csv
  HOA [REST1] HL_E3799
    Saved 260x110 -> HL_E3799_rest1_hoa110roi.csv
  HOA [REST2] HL_E3799
    Saved 260x110 -> HL_E3799_rest2_hoa110roi.csv
  HOA [REST1] TP_E3834
    Saved 260x110 -> TP_E3834_rest1_hoa110roi.csv
  HOA [REST2] TP_E3834
    Saved 260x110 -> TP_E3834_rest2_hoa110roi.csv
  HOA [REST1] LD_E3973
    Saved 260x110 -> LD_E3973_rest1_hoa110roi.csv
  HOA [REST2] LD_E3973
    Saved 260x110 -> LD_E3973_rest2_hoa110roi.csv
  HOA [REST1] SA_E4051
    Saved 260x110 -> SA_E4051_rest1_hoa110roi.csv
  HOA [REST2] SA_E4051
    Saved 260x110 -> SA_E4051_rest2_hoa110roi.csv
  HOA [REST1] LS_E4209
    Saved 260x110 -> LS_E4209_rest1_hoa110roi.csv
  HOA [REST2] LS_E4209
/var/folders/pr/q2gt8qfd2px6jqkj3yd3kzs80000gn/T/ipykernel_41003/2634096490.py:50: UserWarning: 

 full_id session status  n_rois
DP_E3746   rest1     OK     110
DP_E3746   rest2     OK     110
HL_E3799   rest1     OK     110
HL_E3799   rest2     OK     110
TP_E3834   rest1     OK     110
TP_E3834   rest2     OK     110
LD_E3973   rest1     OK     110
LD_E3973   rest2     OK     110
SA_E4051   rest1     OK     110
SA_E4051   rest2     OK     110
LS_E4209   rest1     OK     110
LS_E4209   rest2     OK     110
SA_E4253   rest1     OK     110
SA_E4253   rest2     OK     110
JV_E4324   rest1     OK     110
JV_E4324   rest2     OK     110
SF_E4350   rest1     OK     110
SF_E4350   rest2     OK     110
KS_E4360   rest1     OK     110
KS_E4360   rest2     OK     110
CB_E4484   rest1     OK     110
CB_E4484   rest2     OK     110
CL_E4673   rest1     OK     110
CL_E4673   rest2     OK     110
CC_E4689   rest1     OK     110
CC_E4689   rest2     OK     110
DH_E4696   rest1     OK     110
DH_E4696   rest2     OK     110
CM_E4697   rest1     OK     110
CM_E4697   rest2     OK     110
LA_E4745

## §8 — Signal extraction functions

### Key decisions
| Parameter | Value | Reason |
|-----------|-------|--------|
| `standardize` | `False` | `mou_mvar_analysis.R` handles standardisation |
| `detrend` | `False` | AFNI `afni_proc.py` already detrended |
| `smoothing_fwhm` | `None` | Smoothing applied upstream in AFNI |
| `strategy` | `mean` | Mean signal across parcel voxels |

Censored TRs are NaN-filled rather than removed so that every output CSV has exactly 260 rows, matching the rest1 and rest2 files already used in the mou_mvar pipeline.

In [16]:
def load_bold(brik_path: Path):
    """Load AFNI BRIK as nibabel image. Validates 4D shape."""
    img = nib.load(str(brik_path))
    if img.ndim != 4:
        raise ValueError(f'Expected 4D, got {img.ndim}D: {brik_path}')
    n_vols = img.shape[3]
    if n_vols != N_TRS_EXPECTED:
        log.warning(f'  Unexpected volume count: {n_vols} (expected {N_TRS_EXPECTED})')
    return img


def extract_roi_timeseries(bold_img, atlas_resampled):
    """
    Extract mean ROI time series using NiftiLabelsMasker.
    Returns (timeseries, masker) where timeseries.shape = (n_TRs, n_ROIs).
    """
    masker = NiftiLabelsMasker(
        labels_img=atlas_resampled,
        standardize=False,
        detrend=False,
        smoothing_fwhm=None,
        memory_level=0,
        verbose=0
    )
    timeseries = masker.fit_transform(bold_img)  # (n_TRs, n_ROIs)
    return timeseries, masker


def apply_censoring(timeseries: np.ndarray,
                    censor_vec: np.ndarray | None) -> tuple[np.ndarray, int]:
    """
    Replace censored TR rows with NaN (or drop, depending on NAN_CENSORED_TRS).
    Returns (timeseries_out, n_censored_applied).
    """
    if censor_vec is None:
        return timeseries, 0

    n_trs = timeseries.shape[0]
    if len(censor_vec) != n_trs:
        log.warning(f'  Censor length {len(censor_vec)} != n_TRs {n_trs} — skipping')
        return timeseries, 0

    n_censored = int(np.sum(~censor_vec))
    if NAN_CENSORED_TRS:
        out = timeseries.astype(float)
        out[~censor_vec, :] = np.nan
    else:
        out = timeseries[censor_vec, :]

    return out, n_censored


def build_column_names(masker, atlas_labels: list) -> list:
    """
    Match masker-returned label integers back to atlas label strings.
    Excludes label 0 (background) which NiftiLabelsMasker drops from
    extraction but may still include in masker.labels_.
    """
    returned = [int(x) for x in masker.labels_ if int(x) != 0]
    cols = []
    for li in returned:
        if 1 <= li <= len(atlas_labels):
            cols.append(atlas_labels[li - 1])
        else:
            cols.append(f'ROI_{li}')
    return cols


print('Extraction functions defined.')

Extraction functions defined.


## §9 — Single-subject test run

**Run this before the full batch.** Process one subject × one session to verify the full pipeline end-to-end without committing to the full run.

Check:
1. Output shape is `(260, 219)` or `(260, N)` where N ≤ 219
2. Censor vector length matches n_TRs
3. NaN rows correspond to expected censored TRs
4. ROI labels in CSV header are recognisable Schaefer/Melbourne names

In [17]:
# ── Select a test subject ─────────────────────────────────────────────────
# Change these to any subject present in your data
TEST_SUBJECT_ID = sorted(subjects.keys())[0]   # first subject alphabetically
TEST_SESSION    = 'rest1'

test_info   = subjects[TEST_SUBJECT_ID]
test_folder = test_info.get(TEST_SESSION)
test_fullid = test_info.get('full_id', TEST_SUBJECT_ID)

print(f'Test subject : {TEST_SUBJECT_ID} ({test_fullid})')
print(f'Test session : {TEST_SESSION}')
print(f'Folder       : {test_folder}')

if test_folder is None:
    raise RuntimeError(f'No {TEST_SESSION} folder for {TEST_SUBJECT_ID}')

Test subject : E3746 (DP_E3746)
Test session : rest1
Folder       : data/source/processed rest scans/E3746


In [18]:
# ── Step 1: Find BRIK ─────────────────────────────────────────────────────
brik_path = find_brik_file(test_folder)
print(f'BRIK: {brik_path}')
assert brik_path is not None, 'BRIK file not found — check folder path'

BRIK: data/source/processed rest scans/E3746/errts.DP_E3746_rest.fanaticor+tlrc.BRIK


In [19]:
# ── Step 2: Load BOLD ─────────────────────────────────────────────────────
bold_img = load_bold(brik_path)
print(f'BOLD shape : {bold_img.shape}')
print(f'Voxel size : {bold_img.header.get_zooms()[:3]}')
print(f'Affine:\n{bold_img.affine}')

BOLD shape : (96, 114, 96, 260)
Voxel size : (2.0, 2.0, 2.0)
Affine:
[[   2.   -0.   -0.  -95.]
 [  -0.    2.   -0. -131.]
 [   0.    0.    2.  -77.]
 [   0.    0.    0.    1.]]


In [20]:
# ── Step 3: Read motion and censor files ──────────────────────────────────
censor_vec = read_censor(test_folder)
enorm      = read_enorm(test_folder)
motion     = read_motion(test_folder)

if censor_vec is not None:
    n_censored = int(np.sum(~censor_vec))
    print(f'Censor vector: {len(censor_vec)} TRs, {n_censored} censored '
          f'({n_censored/len(censor_vec):.1%})')
    print(f'Censored TR indices: {np.where(~censor_vec)[0].tolist()}')
else:
    print('No censor file found.')

if enorm is not None:
    print(f'Enorm: mean={enorm.mean():.3f}  p95={np.percentile(enorm,95):.3f} mm/TR')

if motion:
    for k, v in motion.items():
        print(f'Motion {k}: {v:.4f}')

Censor vector: 260 TRs, 26 censored (10.0%)
Censored TR indices: [57, 58, 59, 60, 61, 203, 204, 207, 208, 214, 215, 220, 221, 235, 236, 238, 239, 248, 249, 251, 252, 253, 254, 257, 258, 259]
Enorm: mean=0.074  p95=0.205 mm/TR
Motion demean_mean_abs: 0.0697
Motion demean_max_abs: 0.5356
Motion deriv_mean_abs: 0.0219
Motion deriv_max_abs: 0.4323


In [21]:
# ── Step 4: Resample atlas to BOLD space ──────────────────────────────────
print('Resampling atlas...')
atlas_resampled = image.resample_to_img(
    atlas_img, bold_img, interpolation='nearest'
)
print(f'Atlas resampled shape: {atlas_resampled.shape}')

# Sanity check: parcel labels should be integers 1..219
parcel_ids = np.unique(atlas_resampled.get_fdata().astype(int))
parcel_ids = parcel_ids[parcel_ids > 0]
print(f'Parcels present in resampled atlas: {len(parcel_ids)} '
      f'(expected ≤ {N_ROIS_EXPECTED})')

Resampling atlas...
Atlas resampled shape: (96, 114, 96)
Parcels present in resampled atlas: 216 (expected ≤ 216)


In [22]:
# ── Step 5: Extract ROI time series ───────────────────────────────────────
print('Extracting...')
timeseries, masker = extract_roi_timeseries(bold_img, atlas_resampled)
print(f'Raw timeseries shape: {timeseries.shape}  (TRs × ROIs)')

col_names = build_column_names(masker, atlas_labels)
print(f'Column names assigned: {len(col_names)}')
print(f'First 5 columns: {col_names[:5]}')
print(f'Last  5 columns: {col_names[-5:]}')

Extracting...
Raw timeseries shape: (260, 216)  (TRs × ROIs)
Column names assigned: 216
First 5 columns: ['7Networks_LH_Vis_1', '7Networks_LH_Vis_2', '7Networks_LH_Vis_3', '7Networks_LH_Vis_4', '7Networks_LH_Vis_5']
Last  5 columns: ['Pall-rh', 'NAcc-lh', 'NAcc-rh', 'STN-lh', 'STN-rh']


In [23]:
# ── Step 6: Apply censoring ────────────────────────────────────────────────
ts_censored, n_nan = apply_censoring(timeseries, censor_vec)
print(f'After censoring: {ts_censored.shape}  NaN rows: {n_nan}')

# Check NaN rows align with censor vector
if censor_vec is not None and NAN_CENSORED_TRS:
    nan_trs     = np.where(np.any(np.isnan(ts_censored), axis=1))[0]
    censored_trs = np.where(~censor_vec)[0]
    match = np.array_equal(nan_trs, censored_trs)
    print(f'NaN rows match censor vector: {match}')

After censoring: (260, 216)  NaN rows: 26
NaN rows match censor vector: True


In [24]:
# ── Step 7: Build and display output DataFrame ────────────────────────────
df_test = pd.DataFrame(ts_censored, columns=col_names)
df_test.index.name = 'TR'

print(f'Output shape: {df_test.shape}  (should be 260 × ~219)')
print(f'NaN count: {df_test.isna().sum().sum()} entries '
      f'({df_test.isna().mean().mean():.1%} of values)')

# Preview first few rows and ROIs
df_test.iloc[:5, :6]

Output shape: (260, 216)  (should be 260 × ~219)
NaN count: 5616 entries (10.0% of values)


,7Networks_LH_Vis_1,7Networks_LH_Vis_2,7Networks_LH_Vis_3,7Networks_LH_Vis_4,7Networks_LH_Vis_5,7Networks_LH_Vis_6
TR,,,,,,
0,-0.278983,-0.182940,-0.166722,0.125824,-1.150275,0.477083
1,-0.069178,-0.115554,-0.244487,-0.126520,-0.413721,0.081298
2,-0.269091,0.159296,-0.121043,-0.457138,0.635751,-0.978723
3,-0.197411,0.008908,-0.156122,-0.411885,-0.235756,-0.576796
4,-0.109246,-0.127521,-0.244997,-0.406481,0.038492,-0.786498


In [25]:
# ── Step 8: Save test output ──────────────────────────────────────────────
test_out = OUTPUT_DIR / TEST_SESSION / f'{test_fullid}_{TEST_SESSION}_219roi.csv'
df_test.to_csv(test_out)
print(f'Test output saved: {test_out}')
print('\n=== TEST PASSED ===')

Test output saved: data/parcellated/219roi/rest1/DP_E3746_rest1_219roi.csv

=== TEST PASSED ===


## §10 — Full batch processing

Processes all subjects and sessions. Skips any subject×session that was already successfully saved.

Expected runtime: ~2–4 minutes per subject (atlas resampling dominates). With 19 subjects × 2 sessions ≈ 38 runs, allow ~90 minutes total.

> Set `OVERWRITE = False` to skip already-completed files (safe to re-run after interruption).

In [26]:
OVERWRITE = False   # set True to reprocess already-saved files

sessions_to_run = (['rest1', 'rest2'] if SESSION_FILTER == 'both'
                   else [SESSION_FILTER])

qc_records = []
banner('FULL BATCH PROCESSING')

for sid, info in sorted(subjects.items()):
    full_id = info.get('full_id', sid)

    for sess in sessions_to_run:
        if sess not in info:
            log.warning(f'  {sid}: no {sess} folder — skipping')
            continue

        out_path = OUTPUT_DIR / sess / f'{full_id}_{sess}_219roi.csv'
        if out_path.exists() and not OVERWRITE:
            log.info(f'  {full_id} {sess}: already done — skipping')
            qc_records.append({'subject': sid, 'full_id': full_id,
                                'session': sess, 'status': 'SKIPPED'})
            continue

        log.info(f'\n[{sess.upper()}] {full_id}')
        qc = {'subject': sid, 'full_id': full_id, 'session': sess,
              'status': 'FAIL', 'n_trs': 0, 'n_censored': 0,
              'censor_pct': 0.0, 'n_rois': 0,
              'enorm_mean': np.nan, 'enorm_p95': np.nan, 'notes': ''}

        folder = info[sess]

        try:
            # BOLD
            brik_path = find_brik_file(folder)
            if brik_path is None:
                qc['notes'] = 'No BRIK file'; qc_records.append(qc); continue

            bold_img = load_bold(brik_path)
            qc['n_trs'] = bold_img.shape[3]
            log.info(f'  BOLD: {bold_img.shape}')

            # Motion
            censor_vec = read_censor(folder)
            enorm      = read_enorm(folder)

            if censor_vec is not None:
                n_cens   = int(np.sum(~censor_vec))
                cens_pct = n_cens / qc['n_trs']
                qc['n_censored'] = n_cens
                qc['censor_pct'] = round(cens_pct, 4)
                log.info(f'  Censor: {n_cens}/{qc["n_trs"]} ({cens_pct:.1%})')
                if cens_pct > (1 - MIN_GOOD_TR_FRAC):
                    log.warning(f'  !! Excessive censoring {cens_pct:.1%} — SKIP')
                    qc['notes'] = f'Excessive censoring {cens_pct:.1%}'
                    qc_records.append(qc); continue

            if enorm is not None:
                qc['enorm_mean'] = round(float(np.nanmean(enorm)), 4)
                qc['enorm_p95']  = round(float(np.nanpercentile(enorm, 95)), 4)
                log.info(f'  Enorm mean={qc["enorm_mean"]:.3f}  p95={qc["enorm_p95"]:.3f}')

            # Atlas resample
            atlas_res = image.resample_to_img(
                atlas_img, bold_img, interpolation='nearest'
            )

            # Extract
            ts, masker = extract_roi_timeseries(bold_img, atlas_res)
            qc['n_rois'] = ts.shape[1]
            log.info(f'  Extracted: {ts.shape[0]} TRs × {ts.shape[1]} ROIs')

            # Censor
            ts, _ = apply_censoring(ts, censor_vec)

            # Save 219-ROI CSV
            col_names = build_column_names(masker, atlas_labels)
            df_out = pd.DataFrame(ts, columns=col_names)
            df_out.index.name = 'TR'
            df_out.to_csv(out_path)
            log.info(f'  Saved: {out_path.name}')

            qc['status'] = 'OK'
            qc_records.append(qc)

        except Exception as e:
            log.error(f'  ERROR: {e}')
            qc['notes'] = str(e)
            qc_records.append(qc)

log.info(f'\nBatch complete: {sum(r["status"]=="OK" for r in qc_records)} OK  '
         f'{sum(r["status"]=="FAIL" for r in qc_records)} FAIL  '
         f'{sum(r["status"]=="SKIPPED" for r in qc_records)} skipped')


FULL BATCH PROCESSING
  DP_E3746 rest1: already done — skipping

[REST2] DP_E3746
  BOLD: (96, 114, 96, 260)
  Censor: 5/260 (1.9%)
  Enorm mean=0.055  p95=0.132
  Extracted: 260 TRs × 216 ROIs
  Saved: DP_E3746_rest2_219roi.csv

[REST1] HL_E3799
  BOLD: (96, 114, 96, 260)
  Censor: 33/260 (12.7%)
  Enorm mean=0.091  p95=0.234
  Extracted: 260 TRs × 216 ROIs
  Saved: HL_E3799_rest1_219roi.csv

[REST2] HL_E3799
  BOLD: (96, 114, 96, 260)
  Censor: 8/260 (3.1%)
  Enorm mean=0.063  p95=0.137
  Extracted: 260 TRs × 216 ROIs
  Saved: HL_E3799_rest2_219roi.csv

[REST1] TP_E3834
  BOLD: (96, 114, 96, 260)
  Censor: 0/260 (0.0%)
  Enorm mean=0.043  p95=0.099
  Extracted: 260 TRs × 216 ROIs
  Saved: TP_E3834_rest1_219roi.csv

[REST2] TP_E3834
  BOLD: (96, 114, 96, 260)
  Censor: 22/260 (8.5%)
  Enorm mean=0.060  p95=0.187
  Extracted: 260 TRs × 216 ROIs
  Saved: TP_E3834_rest2_219roi.csv

[REST1] LD_E3973
  BOLD: (96, 114, 96, 260)
  Censor: 9/260 (3.5%)
  Enorm mean=0.057  p95=0.146
  Extract

## §11 — QC summary

Reviews all processed sessions. Reports motion statistics and flags any failures.

In [27]:
qc_df = pd.DataFrame(qc_records)

ok     = qc_df[qc_df.status == 'OK']
failed = qc_df[qc_df.status == 'FAIL']
skipped = qc_df[qc_df.status == 'SKIPPED']

banner('QC SUMMARY')
log.info(f'  Total sessions : {len(qc_df)}')
log.info(f'  OK             : {len(ok)}')
log.info(f'  Skipped        : {len(skipped)}')
log.info(f'  Failed         : {len(failed)}')

if len(failed) > 0:
    log.info('\n  FAILURES:')
    for _, r in failed.iterrows():
        log.info(f'    {r.full_id} {r.session}: {r.notes}')

if len(ok) > 0:
    log.info('\n  Motion summary (OK sessions):')
    log.info(f'    Enorm mean : {ok.enorm_mean.mean():.3f} ± {ok.enorm_mean.std():.3f} mm/TR')
    log.info(f'    Enorm p95  : {ok.enorm_p95.mean():.3f} ± {ok.enorm_p95.std():.3f} mm/TR')
    log.info(f'    Censor pct : {ok.censor_pct.mean():.1%} ± {ok.censor_pct.std():.1%}')
    log.info(f'    ROIs/session: {int(ok.n_rois.mode()[0])} (mode)')

qc_save = OUTPUT_DIR / 'parcellation_qc.csv'
qc_df.to_csv(qc_save, index=False)
log.info(f'\n  QC saved: {qc_save}')

qc_df[['full_id','session','status','n_trs','n_rois',
       'n_censored','censor_pct','enorm_mean','enorm_p95','notes']]


QC SUMMARY
  Total sessions : 46
  OK             : 39
  Skipped        : 1
  Failed         : 6

  FAILURES:
    CL_E4673 rest1: Excessive censoring 42.7%
    CL_E4673 rest2: Excessive censoring 65.4%
    DH_E4696 rest1: Excessive censoring 38.1%
    DH_E4696 rest2: Excessive censoring 49.2%
    KE_E5376 rest1: Excessive censoring 36.9%
    MM_E5653 rest2: Excessive censoring 21.2%

  Motion summary (OK sessions):
    Enorm mean : 0.056 ± 0.020 mm/TR
    Enorm p95  : 0.134 ± 0.058 mm/TR
    Censor pct : 3.5% ± 4.2%
    ROIs/session: 216 (mode)

  QC saved: data/parcellated/219roi/parcellation_qc.csv


,full_id,session,status,n_trs,n_rois,n_censored,censor_pct,enorm_mean,enorm_p95,notes
0,DP_E3746,rest1,SKIPPED,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,DP_E3746,rest2,OK,260.0,216.0,5.0,0.0192,0.0547,0.1318,
2,HL_E3799,rest1,OK,260.0,216.0,33.0,0.1269,0.0912,0.2341,
3,HL_E3799,rest2,OK,260.0,216.0,8.0,0.0308,0.0628,0.1366,
4,TP_E3834,rest1,OK,260.0,216.0,0.0,0.0000,0.0425,0.0990,
5,TP_E3834,rest2,OK,260.0,216.0,22.0,0.0846,0.0604,0.1866,
6,LD_E3973,rest1,OK,260.0,216.0,9.0,0.0346,0.0566,0.1462,
7,LD_E3973,rest2,OK,260.0,216.0,12.0,0.0462,0.0577,0.1432,
8,SA_E4051,rest1,OK,260.0,216.0,0.0,0.0000,0.0265,0.0515,
9,SA_E4051,rest2,OK,260.0,216.0,0.0,0.0000,0.0264,0.0551,


## §12 — Output verification

Spot-check a saved CSV: verify shape, NaN pattern, and signal statistics.
Run this after the batch to confirm outputs are ready for `mou_mvar_analysis.R`.

In [28]:
# Load a saved output CSV and verify it
verify_subject = sorted(subjects.keys())[0]
verify_full_id = subjects[verify_subject].get('full_id', verify_subject)
verify_session = 'rest1'

verify_path = OUTPUT_DIR / verify_session / f'{verify_full_id}_{verify_session}_219roi.csv'
print(f'Verifying: {verify_path}')

if verify_path.exists():
    df_verify = pd.read_csv(verify_path, index_col=0)
    print(f'Shape       : {df_verify.shape}  (expected ~260 × {N_ROIS_EXPECTED})')
    print(f'NaN TRs     : {df_verify.isna().any(axis=1).sum()} rows contain NaN')
    print(f'Signal mean : {df_verify.mean().mean():.4f}  (expected ~0 after detrend)')
    print(f'Signal std  : {df_verify.std().mean():.4f}')
    print(f'ROI columns : {list(df_verify.columns[:4])} ... {list(df_verify.columns[-4:])}')
    print('\nFirst 5 rows × 6 ROIs:')
    display(df_verify.iloc[:5, :6])
else:
    print(f'File not found: {verify_path}')

Verifying: data/parcellated/219roi/rest1/DP_E3746_rest1_219roi.csv
Shape       : (260, 216)  (expected ~260 × 216)
NaN TRs     : 26 rows contain NaN
Signal mean : 0.0000  (expected ~0 after detrend)
Signal std  : 0.2952
ROI columns : ['7Networks_LH_Vis_1', '7Networks_LH_Vis_2', '7Networks_LH_Vis_3', '7Networks_LH_Vis_4'] ... ['NAcc-lh', 'NAcc-rh', 'STN-lh', 'STN-rh']

First 5 rows × 6 ROIs:


,7Networks_LH_Vis_1,7Networks_LH_Vis_2,7Networks_LH_Vis_3,7Networks_LH_Vis_4,7Networks_LH_Vis_5,7Networks_LH_Vis_6
TR,,,,,,
0,-0.278983,-0.182940,-0.166722,0.125824,-1.150275,0.477083
1,-0.069178,-0.115554,-0.244487,-0.126520,-0.413721,0.081298
2,-0.269091,0.159296,-0.121043,-0.457138,0.635751,-0.978723
3,-0.197411,0.008908,-0.156122,-0.411885,-0.235756,-0.576796
4,-0.109246,-0.127521,-0.244997,-0.406481,0.038492,-0.786498


In [29]:
# List all output files
print('=== Generated output files ===')
for sess in ['rest1', 'rest2']:
    files = sorted((OUTPUT_DIR / sess).glob('*.csv'))
    print(f'\n{sess}: {len(files)} files')
    for f in files:
        size_kb = f.stat().st_size / 1024
        print(f'  {f.name}  ({size_kb:.0f} KB)')

=== Generated output files ===

rest1: 20 files
  AC_E5694_rest1_219roi.csv  (1108 KB)
  CB_E4484_rest1_219roi.csv  (1062 KB)
  CC_E4689_rest1_219roi.csv  (945 KB)
  CM_E4697_rest1_219roi.csv  (1124 KB)
  DP_E3746_rest1_219roi.csv  (1010 KB)
  HL_E3799_rest1_219roi.csv  (980 KB)
  IR_E5580_rest1_219roi.csv  (1105 KB)
  JV_E4324_rest1_219roi.csv  (1125 KB)
  KH_E5693_rest1_219roi.csv  (1035 KB)
  KS_E4360_rest1_219roi.csv  (1075 KB)
  LA_E4745_rest1_219roi.csv  (1095 KB)
  LD_E3973_rest1_219roi.csv  (1081 KB)
  LS_E4209_rest1_219roi.csv  (1078 KB)
  MM_E5653_rest1_219roi.csv  (928 KB)
  PS_E5586_rest1_219roi.csv  (1114 KB)
  RS_E5215_rest1_219roi.csv  (1116 KB)
  SA_E4051_rest1_219roi.csv  (1121 KB)
  SA_E4253_rest1_219roi.csv  (1073 KB)
  SF_E4350_rest1_219roi.csv  (1098 KB)
  TP_E3834_rest1_219roi.csv  (1123 KB)

rest2: 20 files
  AC_E5694_rest2_219roi.csv  (1097 KB)
  CB_E4484_rest2_219roi.csv  (1097 KB)
  CC_E4689_rest2_219roi.csv  (1009 KB)
  CM_E4697_rest2_219roi.csv  (1083 KB)
  